# Chapter 17: Algorithmic Fairness and Bias
**Part V — AI Ethics, Governance & Law**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

Sources of bias, fairness definitions, measurement, and mitigation.

## Learning Objectives
See Chapter 17 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Fairness assessment with Fairlearn


In [ ]:
# Fairness assessment with Fairlearn
from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score, recall_score

mf = MetricFrame(
    metrics={"accuracy": accuracy_score, "recall": recall_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_attr   # e.g. gender or race
)

print(mf.by_group)
print("Max accuracy gap:", mf.difference()["accuracy"])
print("Max recall gap:",   mf.difference()["recall"])

## Demonstrate fairness incompatibility (Chouldechova's theorem)


In [ ]:
# Demonstrate fairness incompatibility (Chouldechova's theorem)
import numpy as np
import pandas as pd

np.random.seed(42)

def simulate_recidivism_dataset(n=2000, base_rate_A=0.3, base_rate_B=0.5):
    """Simulate two groups with different recidivism base rates."""
    n_each = n // 2
    # Group A (lower base rate)
    y_A = np.random.binomial(1, base_rate_A, n_each)
    score_A = y_A * np.random.uniform(0.5, 1.0, n_each) + (1-y_A) * np.random.uniform(0.0, 0.6, n_each)
    # Group B (higher base rate)
    y_B = np.random.binomial(1, base_rate_B, n_each)
    score_B = y_B * np.random.uniform(0.5, 1.0, n_each) + (1-y_B) * np.random.uniform(0.0, 0.6, n_each)
    return (np.concatenate([y_A, y_B]),
            np.concatenate([score_A, score_B]),
            np.concatenate([np.zeros(n_each), np.ones(n_each)]))

y, score, group = simulate_recidivism_dataset()
threshold = 0.5
pred = (score >= threshold).astype(int)

def fairness_metrics(y, pred, group, g):
    mask = group == g
    yg, pg = y[mask], pred[mask]
    tp = ((yg==1)&(pg==1)).sum(); fp = ((yg==0)&(pg==1)).sum()
    tn = ((yg==0)&(pg==0)).sum(); fn = ((yg==1)&(pg==0)).sum()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0
    fpr = fp/(fp+tn) if (fp+tn)>0 else 0
    ppv = tp/(tp+fp) if (tp+fp)>0 else 0
    acc = (tp+tn)/len(yg)
    return {'base_rate':yg.mean(), 'tpr':tpr, 'fpr':fpr, 'ppv':ppv, 'accuracy':acc}

ma, mb = fairness_metrics(y, pred, group, 0), fairness_metrics(y, pred, group, 1)

print("Fairness Metric Comparison at threshold = 0.5")
print("=" * 55)
print(f"{'Metric':<18} {'Group A':>12} {'Group B':>12} {'Gap':>10}")
print("-" * 55)
for k in ['base_rate','tpr','fpr','ppv','accuracy']:
    gap = abs(ma[k]-mb[k])
    print(f"{k:<18} {ma[k]:>12.4f} {mb[k]:>12.4f} {gap:>10.4f}")

print("""
Key insight (Chouldechova's theorem):
When base rates differ between groups:
  → Equalising TPR and FPR (equalised odds) requires different thresholds
  → Different thresholds produce different PPV (violates calibration)
  → You CANNOT satisfy both simultaneously — it is mathematically impossible
  
Choosing which criterion to satisfy is an ETHICAL decision, not a technical one.""")

---

## 📚 Review Questions

See Chapter 17 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 17 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*